# Improved FER2013 Emotion Recognition
## Enhanced Model with Multiple Accuracy Improvements

This notebook implements several techniques to improve accuracy over the baseline:
1. **Advanced Data Augmentation** - Mixup, Cutout, Color jitter
2. **Class-Balanced Sampling** - Address imbalanced classes
3. **Deeper CNN Architecture** - With attention mechanisms
4. **Better Optimization** - Cosine annealing with warmup
5. **Test-Time Augmentation (TTA)** - Average predictions across augmentations
6. **Focal Loss** - Handle hard examples better

In [ ]:
# Install required packages
!pip install -q kaggle
!pip install -q opencv-python
!pip install -q timm

In [ ]:
# Upload Kaggle credentials
from google.colab import files
files.upload()

In [ ]:
# Setup Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download and extract dataset
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d fer2013

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Advanced Preprocessing with CLAHE and Augmentations

In [ ]:
# CLAHE Transform for contrast enhancement
class CLAHETransform:
    def __init__(self, clip_limit=3.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img):
        img_np = np.array(img)
        if len(img_np.shape) == 3:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        img_clahe = self.clahe.apply(img_np)
        return Image.fromarray(img_clahe)


# Mixup augmentation
def mixup_data(x, y, alpha=0.2):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


# Cutout regularization
class Cutout:
    def __init__(self, n_holes=1, length=16):
        self.n_holes = n_holes
        self.length = length

    def __call__(self, img):
        h = img.size(1)
        w = img.size(2)
        mask = np.ones((h, w), np.float32)

        for n in range(self.n_holes):
            y = np.random.randint(h)
            x = np.random.randint(w)

            y1 = np.clip(y - self.length // 2, 0, h)
            y2 = np.clip(y + self.length // 2, 0, h)
            x1 = np.clip(x - self.length // 2, 0, w)
            x2 = np.clip(x + self.length // 2, 0, w)

            mask[y1: y2, x1: x2] = 0.

        mask = torch.from_numpy(mask)
        mask = mask.expand_as(img)
        return img * mask


# Training transforms with strong augmentation
train_transform = transforms.Compose([
    CLAHETransform(clip_limit=3.0, tile_grid_size=(8, 8)),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(p=0.6),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    Cutout(n_holes=1, length=16)
])

# Test transforms (no random augmentations)
test_transform = transforms.Compose([
    CLAHETransform(clip_limit=3.0, tile_grid_size=(8, 8)),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# TTA transforms for inference
tta_transforms = [
    transforms.Compose([
        CLAHETransform(clip_limit=3.0, tile_grid_size=(8, 8)),
        transforms.Resize((48, 48)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]),
    transforms.Compose([
        CLAHETransform(clip_limit=3.0, tile_grid_size=(8, 8)),
        transforms.Resize((48, 48)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]),
    transforms.Compose([
        CLAHETransform(clip_limit=3.0, tile_grid_size=(8, 8)),
        transforms.Resize((52, 52)),
        transforms.CenterCrop(48),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
]

In [ ]:
# Load datasets
train_dataset = ImageFolder(root="/content/fer2013/train", transform=train_transform)
test_dataset = ImageFolder(root="/content/fer2013/test", transform=test_transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

# Calculate class weights for balanced sampling
class_counts = np.bincount([label for _, label in train_dataset.imgs])
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for _, label in train_dataset.imgs]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

## Improved CNN Architecture with Attention

In [ ]:
# Squeeze-and-Excitation Attention Block
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


# Improved EmotionCNN with attention and deeper architecture
class ImprovedEmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(ImprovedEmotionCNN, self).__init__()

        # Block 1: 48x48 -> 24x24
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            SEBlock(64),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        # Block 2: 24x24 -> 12x12
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            SEBlock(128),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        # Block 3: 12x12 -> 6x6
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            SEBlock(256),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(256 * 6 * 6, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


# Initialize model
model = ImprovedEmotionCNN().to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Focal Loss for Hard Examples

In [ ]:
# Focal Loss implementation
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


# Calculate class weights for loss
class_counts = np.bincount([label for _, label in train_dataset.imgs])
class_weights = torch.FloatTensor(1.0 / class_counts).to(device)
class_weights = class_weights / class_weights.sum() * len(class_weights)

# Use Focal Loss with class weights
criterion = FocalLoss(gamma=2.0, weight=class_weights, label_smoothing=0.15)

## Training with Cosine Annealing and Warmup

In [ ]:
# Optimizer with weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Cosine annealing scheduler with warmup
total_epochs = 50
warmup_epochs = 5

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=1e-6)


# Warmup function
def warmup_lr_scheduler(optimizer, epoch, warmup_epochs, base_lr):
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr


# Training function with mixup
def train(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Apply mixup
        mixed_images, targets_a, targets_b, lam = mixup_data(images, labels, alpha=0.2)

        optimizer.zero_grad()
        outputs = model(mixed_images)

        # Mixup loss
        loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (lam * predicted.eq(targets_a.data).cpu().sum().item() +
                   (1 - lam) * predicted.eq(targets_b.data).cpu().sum().item())

    return total_loss / len(train_loader), 100 * correct / total


# Evaluation function
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total


# Test-Time Augmentation inference
def evaluate_with_tta(model, dataset, tta_transforms):
    model.eval()
    all_probs = []
    all_labels = []

    for transform in tta_transforms:
        dataset.transform = transform
        loader = DataLoader(dataset, batch_size=64, shuffle=False)

        probs_list = []
        with torch.no_grad():
            for images, labels in loader:
                images = images.to(device)
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                probs_list.append(probs)

        all_probs.append(torch.cat(probs_list, dim=0))

        if not all_labels:
            all_labels = labels

    # Average probabilities
    avg_probs = torch.stack(all_probs).mean(0)
    _, preds = torch.max(avg_probs, 1)

    correct = (preds.cpu() == all_labels).sum().item()
    total = len(all_labels)

    return 100 * correct / total, avg_probs

In [ ]:
# Training loop
train_losses = []
train_accuracies = []
test_accuracies = []
best_accuracy = 0
best_model_state = None

for epoch in range(total_epochs):
    # Warmup
    warmup_lr_scheduler(optimizer, epoch, warmup_epochs, base_lr=0.001)

    # Train
    loss, train_acc = train(model, train_loader, criterion, optimizer, epoch)
    train_losses.append(loss)
    train_accuracies.append(train_acc)

    # Evaluate
    test_acc = evaluate(model, test_loader)
    test_accuracies.append(test_acc)

    # Step scheduler (after warmup)
    if epoch >= warmup_epochs:
        scheduler.step()

    # Save best model
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        best_model_state = model.state_dict().copy()

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d}/{total_epochs} | LR: {current_lr:.6f} | Loss: {loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print(f"\nBest Test Accuracy: {best_accuracy:.2f}%")

In [ ]:
# Load best model and evaluate with TTA
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("Evaluating with Test-Time Augmentation...")
tta_accuracy, tta_probs = evaluate_with_tta(model, test_dataset, tta_transforms)
print(f"Test Accuracy with TTA: {tta_accuracy:.2f}%")
print(f"Test Accuracy without TTA: {evaluate(model, test_loader):.2f}%")

In [ ]:
# Plot training curves
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Training Loss", color="blue")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Over Epochs")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Accuracy", color="orange")
plt.plot(test_accuracies, label="Test Accuracy", color="green")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Over Epochs")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

def get_all_preds(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)

y_pred, y_true = get_all_preds(model, test_loader)

# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.classes)

plt.figure(figsize=(10, 8))
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix on Test Set")
plt.grid(False)
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=train_dataset.classes))

In [ ]:
# Per-class accuracy
import seaborn as sns

num_classes = len(train_dataset.classes)
class_correct = [0] * num_classes
class_total = [0] * num_classes

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        for i in range(len(labels)):
            label = labels[i]
            class_correct[label] += (preds[i] == label).item()
            class_total[label] += 1

acc_per_class = [100 * c / t if t > 0 else 0 for c, t in zip(class_correct, class_total)]
plt.figure(figsize=(10, 5))
sns.barplot(x=train_dataset.classes, y=acc_per_class)
plt.title("Per-Class Accuracy on Test Set")
plt.ylabel("Accuracy (%)")
plt.xlabel("Emotion")
plt.ylim(0, 100)
plt.xticks(rotation=45)
plt.grid(True, axis='y')
plt.show()

print("\nPer-class accuracy:")
for cls, acc in zip(train_dataset.classes, acc_per_class):
    print(f"{cls}: {acc:.2f}%")

In [ ]:
# Prediction demo
emotion_classes = train_dataset.classes

def predict_and_plot(model, image_path, transform):
    img = Image.open(image_path).convert('L')
    img_t = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        output = model(img_t)
        probs = F.softmax(output, dim=1).cpu().numpy().flatten()
        top3_idx = probs.argsort()[-3:][::-1]

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.title("Input Image")

    plt.subplot(1, 2, 2)
    colors = plt.cm.viridis(np.linspace(0, 1, 7))
    bars = plt.bar(range(7), [probs[i] * 100 for i in range(7)], color=colors)
    plt.xticks(range(7), emotion_classes, rotation=45, ha='right')
    plt.ylabel("Probability (%)")
    plt.title(f"Top Prediction: {emotion_classes[top3_idx[0]]} ({probs[top3_idx[0]]*100:.1f}%)")
    plt.ylim(0, 100)

    # Highlight top 3
    for idx in top3_idx:
        bars[idx].set_edgecolor('red')
        bars[idx].set_linewidth(2)

    plt.tight_layout()
    plt.show()


# Upload and predict test images
from google.colab import files
uploaded = files.upload()

for image_path in uploaded.keys():
    predict_and_plot(model, image_path, test_transform)

In [ ]:
# Save the best model
torch.save({
    'model_state_dict': best_model_state,
    'accuracy': best_accuracy,
    'architecture': 'ImprovedEmotionCNN'
}, 'best_emotion_model.pth')

print(f"Model saved! Best accuracy: {best_accuracy:.2f}%")
print(f"With TTA: {tta_accuracy:.2f}%")